# Ch.1 — Statistical Anomaly Detection

> Flag transactions whose features deviate significantly from the normal distribution. The simplest anomaly detector — and the baseline every later method must beat.

**Dataset:** Credit Card Fraud Detection — 284,807 transactions, 0.17% fraud  
**Task:** Score each transaction by statistical deviation, flag anomalies at 0.5% FPR

## The Core Idea

Statistical anomaly detection assigns each observation a **score** measuring deviation from "normal":

| Method | Score | Assumption |
|--------|-------|------------|
| **Z-Score** | $z = (x - \mu) / \sigma$ | Gaussian features |
| **IQR** | Count of features outside $[Q_1 - k \cdot \text{IQR}, Q_3 + k \cdot \text{IQR}]$ | None (distribution-free) |
| **Mahalanobis** | $D_M = \sqrt{(\mathbf{x}-\boldsymbol{\mu})^\top \Sigma^{-1} (\mathbf{x}-\boldsymbol{\mu})}$ | Multivariate Gaussian |

Anomalies live in the tails. Statistics gives us principled ways to measure "tailness."

## Running Example

You're a data scientist at a payment processor. 284,807 transactions, 492 confirmed fraud (0.17%). Your first task: build the cheapest, fastest, most interpretable detector. No neural networks — just statistics.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)
IMG_DIR = "img/"

import os
os.makedirs(IMG_DIR, exist_ok=True)
print("Libraries loaded.")

In [ ]:
# ── Load the Credit Card Fraud dataset ─────────────────────────────────────
# Download from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
df = pd.read_csv("creditcard.csv")
if os.environ.get("CI"):  # limit rows for CI runs
    df = df.head(1000)

print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
print(f"\nFraud rate: {df['Class'].mean():.4%} ({df['Class'].sum()} fraud / {len(df)} total)")
print(f"\nFeatures: Time, V1-V28 (PCA anonymized), Amount")
print(f"Target: Class (0=legitimate, 1=fraud)")
df.head()

In [ ]:
# ── Class imbalance visualization ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df["Class"].value_counts()
axes[0].bar(["Legitimate", "Fraud"], counts.values, color=["#27ae60", "#e74c3c"])
axes[0].set_title("Transaction Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2000, f"{v:,}", ha="center", fontweight="bold")

# Pie chart
axes[1].pie([counts[0], counts[1]], labels=["Legitimate\n99.83%", "Fraud\n0.17%"],
            colors=["#27ae60", "#e74c3c"], startangle=90,
            explode=[0, 0.15], autopct="%1.2f%%")
axes[1].set_title("Extreme Class Imbalance")

plt.tight_layout()
plt.savefig(f"{IMG_DIR}class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()
print("A model predicting 'legitimate' for everything gets 99.83% accuracy — and catches ZERO fraud.")

In [ ]:
# ── Data split (time-based to respect temporal ordering) ───────────────────
# The dataset is ordered by Time — use first 80% for training, last 20% for testing
X = df.drop("Class", axis=1).values
y = df["Class"].values
feature_names = [c for c in df.columns if c != "Class"]

split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Compute statistics on LEGITIMATE training data only
X_normal = X_train[y_train == 0]

print(f"Train: {X_train.shape} ({y_train.sum()} fraud)")
print(f"Test:  {X_test.shape} ({y_test.sum()} fraud)")
print(f"Normal training samples (for fitting statistics): {X_normal.shape}")

## Z-Score Anomaly Detection

$$z_j = \frac{x_j - \mu_j}{\sigma_j}$$

Compute Z-score per feature, then take the **max absolute Z-score** across all features as the anomaly score. Transactions with extreme values on ANY feature get high scores.

In [ ]:
# ── Z-Score scoring ────────────────────────────────────────────────────────
mu = X_normal.mean(axis=0)
sigma = X_normal.std(axis=0) + 1e-8  # avoid division by zero

# Compute Z-scores for test set
z_scores_test = np.abs((X_test - mu) / sigma)
z_max = z_scores_test.max(axis=1)  # max absolute Z across all features

# Evaluate at target FPR = 0.5%
fpr_z, tpr_z, thresh_z = roc_curve(y_test, z_max)
auc_z = auc(fpr_z, tpr_z)

idx_005 = np.where(fpr_z <= 0.005)[0][-1]
recall_z = tpr_z[idx_005]
threshold_z = thresh_z[idx_005]

print(f"Z-Score Results:")
print(f"  AUC: {auc_z:.4f}")
print(f"  Recall @ 0.5% FPR: {recall_z:.2%}")
print(f"  Threshold: {threshold_z:.2f}")

In [ ]:
# ── ROC Curve ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full ROC
axes[0].plot(fpr_z, tpr_z, color="#2980b9", linewidth=2, label=f"Z-Score (AUC={auc_z:.3f})")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random")
axes[0].scatter([fpr_z[idx_005]], [tpr_z[idx_005]], color="#e74c3c", s=100, zorder=5,
               label=f"@ 0.5% FPR: {recall_z:.1%} recall")
axes[0].set(title="ROC Curve — Z-Score Detector", xlabel="False Positive Rate", ylabel="True Positive Rate (Recall)")
axes[0].legend()

# Zoomed ROC (low FPR region)
mask = fpr_z <= 0.05
axes[1].plot(fpr_z[mask], tpr_z[mask], color="#2980b9", linewidth=2)
axes[1].axvline(0.005, color="#e74c3c", linestyle="--", alpha=0.5, label="Target FPR = 0.5%")
axes[1].scatter([fpr_z[idx_005]], [tpr_z[idx_005]], color="#e74c3c", s=100, zorder=5)
axes[1].set(title="ROC Curve (Zoomed to FPR < 5%)", xlabel="FPR", ylabel="Recall")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{IMG_DIR}roc_zscore.png", dpi=150, bbox_inches="tight")
plt.show()

## Feature Discrimination Analysis

Which PCA features best distinguish fraud from legitimate? Features with the largest Z-score separation are the most discriminative.

In [ ]:
# ── Feature discrimination: which features separate fraud best? ────────────
fraud_mask = y_test == 1
mean_z_fraud = z_scores_test[fraud_mask].mean(axis=0)
mean_z_legit = z_scores_test[~fraud_mask].mean(axis=0)
separation = mean_z_fraud - mean_z_legit

# Top 10 most discriminative features
top_idx = np.argsort(np.abs(separation))[-10:][::-1]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e74c3c" if s > 0 else "#2980b9" for s in separation[top_idx]]
ax.barh([feature_names[i] for i in top_idx], np.abs(separation[top_idx]), color=colors)
ax.set(title="Top 10 Most Discriminative Features (|mean Z fraud - mean Z legit|)",
       xlabel="Absolute Z-Score Separation")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(f"{IMG_DIR}feature_discrimination.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Most discriminative: {', '.join([feature_names[i] for i in top_idx[:5]])}")

In [ ]:
# ── Distribution comparison for top features ───────────────────────────────
top_feats = [feature_names[i] for i in top_idx[:4]]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.ravel(), top_feats):
    col_idx = feature_names.index(feat)
    ax.hist(X_test[~fraud_mask, col_idx], bins=80, alpha=0.6, color="#27ae60",
            label="Legitimate", density=True)
    ax.hist(X_test[fraud_mask, col_idx], bins=40, alpha=0.7, color="#e74c3c",
            label="Fraud", density=True)
    ax.set_title(f"{feat} Distribution")
    ax.legend()

plt.suptitle("Feature Distributions: Legitimate vs Fraud", y=1.02)
plt.tight_layout()
plt.savefig(f"{IMG_DIR}feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## Mahalanobis Distance

$$D_M(\mathbf{x}) = \sqrt{(\mathbf{x} - \boldsymbol{\mu})^\top \Sigma^{-1} (\mathbf{x} - \boldsymbol{\mu})}$$

Considers all features jointly, accounting for correlations. For PCA features (uncorrelated), this reduces to the sum of squared Z-scores.

In [ ]:
# ── Mahalanobis distance scoring ───────────────────────────────────────────
# Regularize covariance for numerical stability
cov = np.cov(X_normal.T) + np.eye(X_normal.shape[1]) * 1e-6
cov_inv = np.linalg.inv(cov)

diff = X_test - mu
mahal_scores = np.sqrt(np.sum(diff @ cov_inv * diff, axis=1))

# Evaluate
fpr_m, tpr_m, thresh_m = roc_curve(y_test, mahal_scores)
auc_m = auc(fpr_m, tpr_m)

idx_005_m = np.where(fpr_m <= 0.005)[0][-1]
recall_m = tpr_m[idx_005_m]

print(f"Mahalanobis Distance Results:")
print(f"  AUC: {auc_m:.4f}")
print(f"  Recall @ 0.5% FPR: {recall_m:.2%}")

In [ ]:
# ── Score distributions: normal vs fraud ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scores, title in zip(axes, [z_max, mahal_scores],
                              ["Z-Score (max |z|)", "Mahalanobis Distance"]):
    ax.hist(scores[~fraud_mask], bins=100, alpha=0.6, color="#27ae60",
            label="Legitimate", density=True)
    ax.hist(scores[fraud_mask], bins=50, alpha=0.7, color="#e74c3c",
            label="Fraud", density=True)
    ax.set(title=f"{title} Distribution", xlabel="Score", ylabel="Density")
    ax.legend()

plt.tight_layout()
plt.savefig(f"{IMG_DIR}score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Note: significant overlap between classes — this is why recall is limited.")

In [ ]:
# ── Comparison ROC: Z-Score vs Mahalanobis ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_z, tpr_z, color="#2980b9", linewidth=2, label=f"Z-Score (AUC={auc_z:.3f})")
ax.plot(fpr_m, tpr_m, color="#e74c3c", linewidth=2, label=f"Mahalanobis (AUC={auc_m:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.axvline(0.005, color="gray", linestyle=":", alpha=0.5, label="Target FPR=0.5%")
ax.set(title="ROC Comparison: Statistical Methods",
       xlabel="False Positive Rate", ylabel="Recall")
ax.legend()
plt.tight_layout()
plt.savefig(f"{IMG_DIR}roc_comparison_stats.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nSummary:")
print(f"  Z-Score:      {recall_z:.1%} recall @ 0.5% FPR")
print(f"  Mahalanobis:  {recall_m:.1%} recall @ 0.5% FPR")
print(f"\nTarget: 80% recall. We're at ~45% — need better methods (Ch.2+).")

## Progress Check

| Constraint | Target | Status |
|------------|--------|--------|
| #1 DETECTION | >80% recall | ❌ ~45% — missing more than half |
| #2 PRECISION | <0.5% FPR | ✅ Set by threshold |
| #3 REAL-TIME | <100ms | ✅ Z-scores < 1ms |
| #4 ADAPTABILITY | Drift handling | ❌ Static statistics |
| #5 EXPLAINABILITY | Justifiable | ⚡ Z-scores are interpretable |

**Next**: Ch.2 — Isolation Forest (no distribution assumptions, ~72% recall)

## Exercises

In [ ]:
# ── Exercise 1: IQR-based anomaly detection ────────────────────────────────
# TODO: Implement IQR-based scoring
# For each feature, compute Q1, Q3, IQR from normal training data
# Score each test transaction by counting how many features fall outside
# [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
# Evaluate recall @ 0.5% FPR and compare to Z-score

pass

In [ ]:
# ── Exercise 2: Robust Z-scores using median and MAD ───────────────────────
# TODO: Replace mean/std with median/MAD (Median Absolute Deviation)
# MAD = median(|x - median(x)|)
# Robust Z = (x - median) / (1.4826 * MAD)
# Compare recall @ 0.5% FPR vs standard Z-score

pass

In [ ]:
# ── Exercise 3: Top-k feature selection ────────────────────────────────────
# TODO: Instead of using all 30 features for Z-score,
# select only the top 5 most discriminative features (from the analysis above)
# Recompute max-Z scores using only these 5 features
# Does feature selection improve or hurt recall?

pass